# Unidad 28 — Ordenamiento por fusión (merge sort)

Semana 4 · Clase 1 · Fuente: PDF parte 1, págs. 68–112

[Índice de la semana](../semana-04.ipynb) | [Anterior](u27-burbuja-seleccion-insercion.ipynb) | [Siguiente](u29-ordenamiento-rapido.ipynb)

**Nota sobre el rango:** la portada de la Unidad 29 está en la pág. 113 del PDF y no pertenece a esta unidad.

In [1]:
def verificar(descripcion, obtenido, esperado):
    """Compara el resultado de un ejercicio con el esperado e informa sin detener el notebook."""
    if obtenido == esperado:
        print(f"Correcto: {descripcion}")
    else:
        print(f"Incorrecto: {descripcion}. Se esperaba {esperado!r} y se obtuvo {obtenido!r}")

## Objetivos de la unidad

- Comprender el algoritmo de ordenamiento por fusión y resolver problemas de ordenamiento con él.
- Explicar la diferencia entre el ordenamiento por fusión y los ordenamientos por burbuja, selección e inserción.
- Comprender y explicar la complejidad temporal del ordenamiento por fusión.
- Implementar el ordenamiento por fusión, que divide la lista, ordena cada parte y las vuelve a unir.
- Implementar una función de fusión que convierta dos listas ordenadas en una lista ordenada.
- Entender que el ordenamiento por fusión es un algoritmo de divide y vencerás que usa funciones recursivas.

**Conocimientos previos (según el PDF):** operadores de comparación, llamadas recursivas que dividen el problema y notación Big O.

**Palabras clave:** problema de ordenamiento, ordenamiento por fusión, fusionar.

**Referencia:** PDF parte 1, págs. 69–71

## Contexto

En la unidad anterior se estudiaron algoritmos de ordenamiento de complejidad O(N²). El PDF plantea que se puede crear un algoritmo más eficiente.

El problema del mundo real: hay 4 canicas de hierro azules y 4 verdes, con igual forma y tamaño pero distinto peso. Las azules pesan 10, 20, 40 y 80 y las verdes 30, 50, 60 y 70 (cada color ya está ordenado por peso). Con una balanza hay que obtener las 8 canicas en orden ascendente.

La balanza permite una sola operación: tomar la canica más ligera de cada uno de los dos barriles, compararlas y colocar la más ligera en un barril nuevo. Como cada barril ya está ordenado, la más ligera de cada uno siempre es la primera. La pregunta de la misión es cuántas comparaciones hacen falta con esta estrategia; la respuesta se calcula más abajo.

**Referencia:** PDF parte 1, págs. 73–74

## 1. Fusionar dos listas ordenadas

Fusionar (merge) significa construir una lista ordenada a partir de dos listas que ya están ordenadas. La estrategia es la de la balanza:

1. Se comparan los primeros elementos de ambas listas (los menores de cada una).
2. El menor se añade a la lista resultado y se descarta de su lista de origen.
3. Se repite hasta que una de las dos listas queda vacía.
4. Los elementos restantes de la otra lista ya están ordenados y se añaden al final.

El PDF lo ilustra con `[10, 12, 20, 27]` y `[13, 15, 22, 25]`: se añaden 10, 12, 13, 15, 20, 22, 25 y, al quedar vacía una lista, se añade el 27 restante.

**Referencia:** PDF parte 1, págs. 83–91

In [2]:
def fusion_con_pasos(izq, der):
    """Fusiona dos listas ordenadas mostrando cada comparación (solo didáctico, no modifica los argumentos)."""
    izq, der = list(izq), list(der)
    resultado = []
    comparaciones = 0
    while izq and der:
        comparaciones += 1
        if izq[0] <= der[0]:
            print(f"  compara {izq[0]} y {der[0]}: se añade {izq[0]}")
            resultado.append(izq.pop(0))
        else:
            print(f"  compara {izq[0]} y {der[0]}: se añade {der[0]}")
            resultado.append(der.pop(0))
    print(f"  una lista quedó vacía; se añaden los restantes: {izq + der}")
    resultado.extend(izq)
    resultado.extend(der)
    return resultado, comparaciones

res, n_comp = fusion_con_pasos([10, 12, 20, 27], [13, 15, 22, 25])
print("Resultado:", res, "| comparaciones:", n_comp)

# La misión de las canicas: azules [10, 20, 40, 80] y verdes [30, 50, 60, 70]
res, n_comp = fusion_con_pasos([10, 20, 40, 80], [30, 50, 60, 70])
print("Canicas:", res, "| comparaciones:", n_comp)

  compara 10 y 13: se añade 10
  compara 12 y 13: se añade 12
  compara 20 y 13: se añade 13
  compara 20 y 15: se añade 15
  compara 20 y 22: se añade 20
  compara 27 y 22: se añade 22
  compara 27 y 25: se añade 25
  una lista quedó vacía; se añaden los restantes: [27]
Resultado: [10, 12, 13, 15, 20, 22, 25, 27] | comparaciones: 7
  compara 10 y 30: se añade 10
  compara 20 y 30: se añade 20
  compara 40 y 30: se añade 30
  compara 40 y 50: se añade 40
  compara 80 y 50: se añade 50
  compara 80 y 60: se añade 60
  compara 80 y 70: se añade 70
  una lista quedó vacía; se añaden los restantes: [80]
Canicas: [10, 20, 30, 40, 50, 60, 70, 80] | comparaciones: 7


**Ampliación (no está en el PDF):** el número de comparaciones al fusionar listas de tamaños `a` y `b` está entre `min(a, b)` y `a + b - 1`. Cada comparación coloca exactamente un elemento, y el último elemento no necesita comparación. Por eso fusionar es una operación de coste lineal O(a + b).

**Nota sobre el PDF:** en la diapositiva de la pág. 90 el texto repite "Como 22 es menor que 27, 22 se añade", igual que la pág. 89. En la imagen de esa página se comparan 27 y 25, y se añade el 25. Es un error de la traducción o de la copia de la diapositiva.

**Clave:** si ambas listas están ordenadas, el menor de todos es siempre uno de los dos primeros elementos.

**Error común:** olvidar añadir los elementos sobrantes cuando una de las listas se vacía antes que la otra.

**Para el quiz:** fusionar dos listas ordenadas de tamaños a y b cuesta O(a + b).

**Referencia:** PDF parte 1, págs. 83–91

## 2. La idea del ordenamiento por fusión

El ordenamiento por fusión divide una lista no ordenada en dos sublistas, las ordena y las fusiona en una lista ordenada. Como las sublistas tampoco están ordenadas, el mismo procedimiento se aplica recursivamente a cada una hasta llegar a listas de un elemento, que por definición ya están ordenadas.

El PDF usa `[27, 10, 12, 20, 25, 13, 15, 22]`:

- Se divide en `[27, 10, 12, 20]` y `[25, 13, 15, 22]`.
- Cada mitad se divide de nuevo, hasta tener 8 listas de un elemento.
- Se fusionan de a pares: listas de 2 elementos, luego de 4 y finalmente una de 8: `[10, 12, 13, 15, 20, 22, 25, 27]`.

**Referencia:** PDF parte 1, págs. 76–82

In [3]:
def mostrar_niveles(lista):
    """Muestra por niveles cómo se divide la lista hasta llegar a elementos sueltos."""
    nivel = [lista]
    n_nivel = 0
    while any(len(parte) > 1 for parte in nivel):
        print(f"División {n_nivel}:", nivel)
        siguiente = []
        for parte in nivel:
            if len(parte) > 1:
                mid = len(parte) // 2
                siguiente.extend([parte[:mid], parte[mid:]])
            else:
                siguiente.append(parte)
        nivel = siguiente
        n_nivel += 1
    print(f"División {n_nivel}:", nivel)

mostrar_niveles([27, 10, 12, 20, 25, 13, 15, 22])

División 0: [[27, 10, 12, 20, 25, 13, 15, 22]]
División 1: [[27, 10, 12, 20], [25, 13, 15, 22]]
División 2: [[27, 10], [12, 20], [25, 13], [15, 22]]
División 3: [[27], [10], [12], [20], [25], [13], [15], [22]]


**Clave:** el caso base es la lista de 0 o 1 elementos, que ya está ordenada. Todo el trabajo real ocurre al fusionar, no al dividir.

**Error común:** pensar que dividir ordena algo. Dividir solo reduce el tamaño del problema.

**Para el quiz:** el ordenamiento por fusión es un algoritmo de divide y vencerás que usa recursión.

**Referencia:** PDF parte 1, págs. 76–82

## 3. Implementación con memoria adicional: `mergesort1` y `merge1`

El método de diseño "dividir y conquistar" se estudiará de nuevo en la Unidad 31. La versión sencilla del PDF crea dos listas nuevas `L` y `R` con cortes (slicing) en cada llamada:

- Si `len(S) > 1`, se calcula la mitad `mid`, se crean `L = S[:mid]` y `R = S[mid:]` (listas separadas, copias), se ordenan recursivamente y se fusionan de vuelta en `S`.
- Si `len(S) <= 1`, no se hace nada porque ya está ordenada.

La función `merge1(S, L, R)` escribe en `S`, posición a posición (índice `k`), el menor entre los primeros elementos de `L` y `R`; al terminar el bucle, copia lo que quede de `L` o de `R`.

El PDF numera las líneas de `mergesort1` del 1 al 9 (incluida la línea `print(S)`) y las de `merge1` del 1 al 14.

**Referencia:** PDF parte 1, págs. 93–98

In [4]:
def merge1(S, L, R):
    k = 0
    while len(L) > 0 and len(R) > 0:
        if L[0] <= R[0]:
            S[k] = L.pop(0)
        else:
            S[k] = R.pop(0)
        k += 1
    while len(L) != 0:
        S[k] = L.pop(0)
        k += 1
    while len(R) != 0:
        S[k] = R.pop(0)
        k += 1


def mergesort1(S):
    n = len(S)
    if n > 1:
        print(S)                 # muestra cada llamada (como en el PDF)
        mid = n // 2
        L, R = S[:mid], S[mid:]  # listas nuevas: memoria adicional
        mergesort1(L)
        mergesort1(R)
        merge1(S, L, R)


S = [27, 10, 12, 20, 25, 13, 15, 22]
mergesort1(S)
print(S)

[27, 10, 12, 20, 25, 13, 15, 22]
[27, 10, 12, 20]
[27, 10]
[12, 20]
[25, 13, 15, 22]
[25, 13]
[15, 22]
[10, 12, 13, 15, 20, 22, 25, 27]


**Nota sobre el PDF:** en la pág. 99 el texto dice que `print(S)` se imprime "en la cuarta línea" de `mergesort1()`, lo que coincide con el listado (línea 4, dentro del `if`). Las siete primeras líneas impresas son las listas que se van dividiendo (una por llamada con más de un elemento) y la última es el resultado ordenado. La salida de la ejecución anterior coincide con la del PDF.

**Clave:** `mergesort1` ordena `S` en el sitio (modifica la lista recibida y no devuelve nada), pero usa copias `L` y `R` en cada llamada.

**Error común:** usar `L.pop(0)` creyendo que es gratis: eliminar el primer elemento de una lista de Python desplaza el resto (coste lineal). Es válido y claro para estudiar, pero se puede evitar con índices (ver la sección 5).

**Referencia:** PDF parte 1, págs. 93–99

## 4. Un paso más: la memoria adicional

El PDF explica que `mergesort1()` crea nuevas listas `L` y `R` en cada llamada; sumadas, tienen el tamaño de la `S` recibida. En las llamadas recursivas, cada nivel trabaja con listas de aproximadamente la mitad del tamaño. El PDF concluye que se usa una memoria adicional de tamaño 2N (el doble del tamaño N de `S`) y que el algoritmo puede modificarse para usar solo N.

**Ampliación (no está en el PDF):** se puede comprobar el 2N contando cuántos elementos están copiados a la vez. Las listas de los antecesores siguen vivas mientras se recursa, así que en el peor momento hay `N + N/2 + N/4 + ...` elementos, que es menor que 2N.

**Referencia:** PDF parte 1, pág. 100

In [5]:
vivos = 0
pico = 0

def mergesort1_medido(S):
    """Igual que mergesort1, pero registra el pico de elementos copiados simultáneamente."""
    global vivos, pico
    n = len(S)
    if n > 1:
        mid = n // 2
        L, R = S[:mid], S[mid:]
        vivos += n                      # L y R juntas tienen n elementos
        pico = max(pico, vivos)
        mergesort1_medido(L)
        mergesort1_medido(R)
        merge1(S, L, R)
        vivos -= n

for N in (8, 16, 64, 1024):
    vivos, pico = 0, 0
    mergesort1_medido(list(range(N, 0, -1)))
    print(f"N = {N:5d} | pico de elementos adicionales = {pico:5d} | 2N = {2 * N}")

N =     8 | pico de elementos adicionales =    14 | 2N = 16
N =    16 | pico de elementos adicionales =    30 | 2N = 32
N =    64 | pico de elementos adicionales =   126 | 2N = 128
N =  1024 | pico de elementos adicionales =  2046 | 2N = 2048


**Clave:** la versión con cortes usa memoria adicional de orden O(N); el valor exacto del pico queda por debajo de 2N.

**Para el quiz:** `mergesort1()` usa memoria adicional proporcional al tamaño de la lista (2N según el PDF); `mergesort2()` evita crear `L` y `R` en cada llamada.

**Referencia:** PDF parte 1, pág. 100

## 5. Implementación más eficiente: `mergesort2` y `merge2`

La mejora del PDF no divide `S` en `L` y `R`: ordena `S` indicando el rango con los índices `low` y `high`.

`mergesort2(S, low, high)`:

- Solo recurre si `low < high`; en caso contrario termina.
- Calcula `mid = (low + high) // 2`, ordena `S[low..mid]` y `S[mid+1..high]` con llamadas recursivas.
- Llama a `merge2(S, low, mid, high)` para fusionar los dos rangos ya ordenados.

`merge2` recorre los dos rangos con los índices `i` (desde `low`) y `j` (desde `mid + 1`), guarda el menor en una lista auxiliar `R`, copia los sobrantes y por último vuelca `R` en `S[low..high]`.

**Referencia:** PDF parte 1, págs. 101–105

In [6]:
def merge2(S, low, mid, high):
    R = []
    i, j = low, mid + 1
    while i <= mid and j <= high:
        if S[i] < S[j]:
            R.append(S[i]); i += 1
        else:
            R.append(S[j]); j += 1
    if i > mid:
        for k in range(j, high + 1):
            R.append(S[k])
    else:
        for k in range(i, mid + 1):
            R.append(S[k])
    for k in range(len(R)):
        S[low + k] = R[k]


def mergesort2(S, low, high):
    if low < high:
        print(S)                 # observar el proceso paso a paso (línea 3 del PDF)
        mid = (low + high) // 2
        mergesort2(S, low, mid)
        mergesort2(S, mid + 1, high)
        merge2(S, low, mid, high)


S = [27, 10, 12, 20, 25, 13, 15, 22]
mergesort2(S, 0, len(S) - 1)
print(S)

[27, 10, 12, 20, 25, 13, 15, 22]
[27, 10, 12, 20, 25, 13, 15, 22]
[27, 10, 12, 20, 25, 13, 15, 22]
[10, 27, 12, 20, 25, 13, 15, 22]
[10, 12, 20, 27, 25, 13, 15, 22]
[10, 12, 20, 27, 25, 13, 15, 22]
[10, 12, 20, 27, 13, 25, 15, 22]
[10, 12, 13, 15, 20, 22, 25, 27]


**Nota sobre el PDF:** el listado de `merge2` usa `S[i] < S[j]` (estricto), mientras que `merge1` usa `L[0] <= R[0]`. Con `<` los empates toman primero el elemento de la derecha, así que `merge2` tal como está **no es estable** (no conserva el orden original de elementos iguales). Con números enteros no se nota; sí importa al ordenar registros. Con `<=` sería estable (se verifica en la sección de ampliación y en el ejercicio 7).

**Nota sobre el PDF:** el PDF resume las líneas 9–14 de `merge2` en dos diapositivas con "Línea 9~11" y "Línea 12~14", y en la segunda dice "hasta que el índice de i sea mid" para el recorrido de `i`. El código real es `range(i, mid + 1)`, es decir, incluye `mid`.

**Clave:** a diferencia de `mergesort1`, el tamaño de `S` no cambia durante la ejecución (el PDF lo destaca en la pág. 106); solo se reordenan sus elementos dentro de los rangos. Esto se ve en la salida: todas las líneas impresas tienen 8 elementos.

**Error común:** llamar `mergesort2(S)` sin los índices. La firma exige `low` y `high`; la llamada inicial es `mergesort2(S, 0, len(S) - 1)`.

**Referencia:** PDF parte 1, págs. 101–106

### Cuestionario del PDF: cuántas veces se ejecuta `merge2()` (prueba sorpresa)

La pág. 108 pide contar las veces que se ejecuta `merge2()` al ordenar dos listas: `[6, 2, 11, 7, 5, 4, 8, 16, 10, 3]` (10 elementos) y `[6, 2, 11, 7, 5, 4, 8, 16, 10, 3, 1, 12, 9]` (13 elementos). Se puede contar ejecutándolo.

**Referencia:** PDF parte 1, pág. 108

In [7]:
llamadas_merge2 = 0

def merge2_contado(S, low, mid, high):
    global llamadas_merge2
    llamadas_merge2 += 1
    merge2(S, low, mid, high)

def mergesort2_contado(S, low, high):
    if low < high:
        mid = (low + high) // 2
        mergesort2_contado(S, low, mid)
        mergesort2_contado(S, mid + 1, high)
        merge2_contado(S, low, mid, high)

for S in ([6, 2, 11, 7, 5, 4, 8, 16, 10, 3], [6, 2, 11, 7, 5, 4, 8, 16, 10, 3, 1, 12, 9]):
    llamadas_merge2 = 0
    mergesort2_contado(S, 0, len(S) - 1)
    print(f"{len(S)} elementos -> merge2 se ejecutó {llamadas_merge2} veces; resultado {S}")

10 elementos -> merge2 se ejecutó 9 veces; resultado [2, 3, 4, 5, 6, 7, 8, 10, 11, 16]
13 elementos -> merge2 se ejecutó 12 veces; resultado [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 16]


**Respuesta:** 9 veces para la primera lista y 12 para la segunda. En general, para N elementos se ejecuta `merge2` exactamente N - 1 veces: cada fusión reduce en uno el número de listas, y se parte de N listas de un elemento hasta llegar a una sola. (Esta justificación es una **Ampliación (no está en el PDF):**; el PDF solo plantea la pregunta.)

**Para el quiz:** número de fusiones en merge sort = N - 1.

**Referencia:** PDF parte 1, pág. 108

## 6. Complejidad temporal O(N log N)

Uno de los objetivos del PDF es explicar la complejidad temporal del ordenamiento por fusión, pero las diapositivas de la unidad no desarrollan el análisis; solo lo piden como objetivo (pág. 69) y lo contrastan con los algoritmos O(N²) de la unidad anterior (pág. 73).

**Ampliación (no está en el PDF):** el razonamiento estándar es el siguiente.

- Dividir por la mitad repetidamente produce unos log₂ N niveles de recursión.
- En cada nivel, las fusiones procesan en total los N elementos, con coste lineal.
- Por tanto el coste total es N por log N, es decir, O(N log N), tanto en el mejor como en el peor caso. Frente a O(N²), la diferencia crece rápido con N.

Se puede comprobar contando comparaciones.

**Referencia:** PDF parte 1, págs. 69 y 73 (objetivo y motivación); el análisis es ampliación

In [8]:
import math

comparaciones = 0

def mergesort_contando(S):
    """Devuelve una lista nueva ordenada y cuenta las comparaciones entre elementos."""
    global comparaciones
    if len(S) <= 1:
        return S
    mid = len(S) // 2
    izq = mergesort_contando(S[:mid])
    der = mergesort_contando(S[mid:])
    resultado = []
    i = j = 0
    while i < len(izq) and j < len(der):
        comparaciones += 1
        if izq[i] <= der[j]:
            resultado.append(izq[i]); i += 1
        else:
            resultado.append(der[j]); j += 1
    resultado.extend(izq[i:])
    resultado.extend(der[j:])
    return resultado

import random
random.seed(42)  # semilla fija para reproducir los datos

print(f"{'N':>6} {'comparaciones':>14} {'N*log2(N)':>10} {'N*N/2':>10}")
for N in (8, 64, 512, 4096):
    datos = [random.randint(0, 10_000) for _ in range(N)]
    comparaciones = 0
    ordenada = mergesort_contando(datos)
    assert ordenada == sorted(datos)
    print(f"{N:>6} {comparaciones:>14} {N * math.log2(N):>10.0f} {N * N // 2:>10}")

     N  comparaciones  N*log2(N)      N*N/2
     8             16         24         32
    64            307        384       2048
   512           3944       4608     131072
  4096          43967      49152    8388608


Para comparar con los algoritmos de la unidad anterior, se mide el tiempo de una burbuja contra `mergesort2`. **Las mediciones de tiempo son orientativas:** dependen de la máquina y de lo que esté ejecutándose; lo importante es la tendencia, no los valores exactos.

**Ampliación (no está en el PDF):**

In [9]:
import random, time, sys
random.seed(42)
sys.setrecursionlimit(10_000)

def burbuja(S):
    S = list(S)
    for fin in range(len(S) - 1, 0, -1):
        for i in range(fin):
            if S[i] > S[i + 1]:
                S[i], S[i + 1] = S[i + 1], S[i]
    return S

def medir(funcion, datos):
    inicio = time.perf_counter()
    funcion(datos)
    return time.perf_counter() - inicio

def ordenar_mergesort2(datos):
    copia = list(datos)
    mergesort2_silencioso(copia, 0, len(copia) - 1)
    return copia

def mergesort2_silencioso(S, low, high):
    if low < high:
        mid = (low + high) // 2
        mergesort2_silencioso(S, low, mid)
        mergesort2_silencioso(S, mid + 1, high)
        merge2(S, low, mid, high)

print("Tiempos orientativos (segundos), semilla 42:")
for N in (500, 1000, 2000):
    datos = [random.randint(0, 100_000) for _ in range(N)]
    t_b = medir(burbuja, datos)
    t_m = medir(ordenar_mergesort2, datos)
    print(f"N = {N:5d} | burbuja ~ {t_b:.4f} | mergesort2 ~ {t_m:.4f}")

Tiempos orientativos (segundos), semilla 42:
N =   500 | burbuja ~ 0.0031 | mergesort2 ~ 0.0003
N =  1000 | burbuja ~ 0.0132 | mergesort2 ~ 0.0006
N =  2000 | burbuja ~ 0.0543 | mergesort2 ~ 0.0013


**Clave:** O(N log N) frente a O(N²): al duplicar N, la burbuja tarda cerca de 4 veces más y el ordenamiento por fusión cerca de 2 veces más.

**Error común:** creer que merge sort es siempre más rápido en cualquier tamaño. Con listas muy pequeñas, la sobrecarga de las llamadas recursivas puede igualar o superar a los métodos simples.

**Para el quiz:** la complejidad temporal del ordenamiento por fusión es O(N log N).

**Referencia:** PDF parte 1, págs. 69 y 73 (el análisis detallado es ampliación)

## 7. Ampliación: estabilidad y comparación con `<` o `<=`

**Ampliación (no está en el PDF):** un ordenamiento es estable si conserva el orden relativo de los elementos con igual clave. En una fusión, al haber empate, el elemento de la izquierda debe salir primero (condición `<=`). Se comprueba con pares `(clave, etiqueta)` ordenando solo por clave.

In [10]:
def fusionar_por_clave(izq, der, estable=True):
    """Fusiona dos listas de pares (clave, etiqueta) ordenadas por clave."""
    resultado, i, j = [], 0, 0
    while i < len(izq) and j < len(der):
        if (izq[i][0] <= der[j][0]) if estable else (izq[i][0] < der[j][0]):
            resultado.append(izq[i]); i += 1
        else:
            resultado.append(der[j]); j += 1
    return resultado + izq[i:] + der[j:]

izq = [(1, "a-izq"), (2, "b-izq")]
der = [(1, "c-der"), (2, "d-der")]
print("Con <= :", fusionar_por_clave(izq, der, estable=True))
print("Con <  :", fusionar_por_clave(izq, der, estable=False))

Con <= : [(1, 'a-izq'), (1, 'c-der'), (2, 'b-izq'), (2, 'd-der')]
Con <  : [(1, 'c-der'), (1, 'a-izq'), (2, 'd-der'), (2, 'b-izq')]


**Clave:** con `<=` los empates conservan el orden original (estable); con `<` se invierte el orden entre las dos mitades.

**Referencia:** ampliación; el PDF usa `<=` en `merge1` y `<` en `merge2` (págs. 96 y 102).

## Proyecto de la unidad

**Problema (PDF, pág. 112):** dadas N listas ordenadas como entrada, escribir un programa que las fusione en una única lista ordenada.

El PDF presenta este problema en la sección de programación en pareja de la unidad; no incluye una solución ni otros proyectos (por ejemplo, la verificación de anagramas no aparece en esta unidad).

**Plan de programación** (propuesto a partir de lo visto, no es un plan del PDF):

1. Reutilizar la fusión de dos listas ordenadas.
2. Fusionar las N listas de a pares, de forma repetida, hasta que quede una sola. Es exactamente la fase de fusión del merge sort: cada ronda reduce a la mitad el número de listas.
3. Manejar los casos límite: ninguna lista, una sola lista, listas vacías dentro de la entrada.

**Referencia:** PDF parte 1, pág. 112

In [11]:
def fusionar_dos(izq, der):
    """Fusiona dos listas ordenadas en una lista nueva ordenada."""
    resultado = []
    i = j = 0
    while i < len(izq) and j < len(der):
        if izq[i] <= der[j]:
            resultado.append(izq[i]); i += 1
        else:
            resultado.append(der[j]); j += 1
    resultado.extend(izq[i:])
    resultado.extend(der[j:])
    return resultado


def fusionar_listas(listas):
    """Fusiona N listas ordenadas en una sola lista ordenada, de a pares por rondas."""
    if not listas:
        return []
    pendientes = [list(lista) for lista in listas]
    while len(pendientes) > 1:
        siguiente = []
        for i in range(0, len(pendientes) - 1, 2):
            siguiente.append(fusionar_dos(pendientes[i], pendientes[i + 1]))
        if len(pendientes) % 2 == 1:      # sobra una lista: pasa a la siguiente ronda
            siguiente.append(pendientes[-1])
        pendientes = siguiente
    return pendientes[0]


entrada = [[1, 4, 9], [2, 3, 10], [0, 5, 6, 7], [], [8]]
resultado = fusionar_listas(entrada)
print("Entrada :", entrada)
print("Resultado:", resultado)
verificar("proyecto: 5 listas", resultado, sorted(sum(entrada, [])))
verificar("proyecto: sin listas", fusionar_listas([]), [])
verificar("proyecto: una lista", fusionar_listas([[3, 5]]), [3, 5])

Entrada : [[1, 4, 9], [2, 3, 10], [0, 5, 6, 7], [], [8]]
Resultado: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10]
Correcto: proyecto: 5 listas
Correcto: proyecto: sin listas
Correcto: proyecto: una lista


### Versión interactiva

La entrada se pide por consola: una lista por línea, con números separados por espacios, y una línea vacía para terminar. Si el usuario cancela (la entrada está vacía o es `None`), el programa termina y procesa lo recibido hasta ese momento.

In [12]:
def leer_listas_interactivo(leer=input):
    """Lee listas ordenadas de la consola hasta recibir una línea vacía o None (cancelar)."""
    listas = []
    while True:
        try:
            linea = leer("Lista ordenada (números separados por espacios; vacío para terminar): ")
        except EOFError:
            break
        if linea is None or linea.strip() == "":
            break
        try:
            listas.append([int(x) for x in linea.split()])
        except ValueError:
            print("Entrada no válida; escribe solo números enteros.")
    return listas


# Versión no interactiva: se simula la entrada con una lista de comandos de prueba.
comandos = iter(["1 4 9", "2 3 10", "0 5 6 7", "abc", "8", ""])
listas = leer_listas_interactivo(leer=lambda mensaje: next(comandos))
print("Listas leídas:", listas)
print("Fusionada    :", fusionar_listas(listas))

# Para usarlo de forma interactiva, quita el comentario de la siguiente línea
# print(fusionar_listas(leer_listas_interactivo()))

Entrada no válida; escribe solo números enteros.
Listas leídas: [[1, 4, 9], [2, 3, 10], [0, 5, 6, 7], [8]]
Fusionada    : [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10]


**Resultado esperado:** la lista `[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10]`. La línea `abc` se rechaza y se vuelve a pedir una lista.

**Ampliación (no está en el PDF):** con k listas y N elementos en total, la fusión por rondas cuesta O(N log k), porque hay log k rondas y cada una procesa N elementos.

**Referencia:** PDF parte 1, pág. 112

### Programación en pareja

El PDF dedica dos diapositivas, en inglés y español, a la práctica de programación en pareja. Resumen de lo que indican:

- **Pautas, mecanismos y plan de contingencia:** establecer pautas para emparejar y mantener las parejas; por ejemplo, turnarse para manejar el ratón. Prever qué hacer si uno de los socios falta o decide no participar, dejando claro que el estudiante activo no será sancionado.
- **Emparejar con habilidades similares, no necesariamente iguales:** un emparejamiento muy desigual suele llevar a una participación desbalanceada. La programación en pareja no es una estrategia de divide y vencerás, sino un esfuerzo de colaboración real en todo el proyecto.
- **Motivar con incentivos extra** y, si hace falta, pedir la pareja solo en una o dos tareas.
- **Evitar trampas en la colaboración:** diseñar y evaluar el curso de modo que se pueda valorar el resultado individual, explicar qué conductas se considerarán trampa y hacer las tareas significativas.
- **Entorno de aprendizaje colaborativo:** actividades formales o informales en las que el docente pide trabajar juntos, distintas de proyectos en que cada uno resuelve solo una parte.

El ejercicio de la pág. 112 (fusionar N listas ordenadas) es el que se propone para esta práctica; su resolución está arriba y en el ejercicio 6 de la Práctica.

**Nota sobre el PDF:** las diapositivas 110 y 111 mezclan el texto en inglés y en español en el mismo párrafo (la extracción de texto los intercala). El contenido anterior es el resumen del texto en español.

**Referencia:** PDF parte 1, págs. 110–112

## Práctica

Los ejercicios 1 a 8 van de menor a mayor dificultad. Los ejercicios 3 y 6 reutilizan `fusionar` del ejercicio 1: complétalo primero. Las celdas sin resolver imprimen "Incorrecto", y es lo esperado antes de escribir la solución.

### Ejercicio 1 — Fusionar dos listas ordenadas

Escribe `fusionar(izq, der)` que reciba dos listas ordenadas de menor a mayor y devuelva una lista nueva con todos sus elementos ordenados. No uses `sorted` ni `sort`.

In [13]:
def fusionar(izq, der):
    # Escribe tu solución aquí
    pass

In [14]:
verificar("fusión intercalada", fusionar([10, 12, 20, 27], [13, 15, 22, 25]), [10, 12, 13, 15, 20, 22, 25, 27])
verificar("una lista vacía", fusionar([], [1, 2]), [1, 2])
verificar("con repetidos", fusionar([1, 3, 3], [3, 4]), [1, 3, 3, 3, 4])

Incorrecto: fusión intercalada. Se esperaba [10, 12, 13, 15, 20, 22, 25, 27] y se obtuvo None
Incorrecto: una lista vacía. Se esperaba [1, 2] y se obtuvo None
Incorrecto: con repetidos. Se esperaba [1, 3, 3, 3, 4] y se obtuvo None


<details><summary>Ver solución</summary>

```python
def fusionar(izq, der):
    resultado = []
    i = j = 0
    while i < len(izq) and j < len(der):
        if izq[i] <= der[j]:
            resultado.append(izq[i])
            i += 1
        else:
            resultado.append(der[j])
            j += 1
    resultado.extend(izq[i:])
    resultado.extend(der[j:])
    return resultado
```

Se avanza con un índice en cada lista, se toma el menor y al final se copian los sobrantes.
</details>

### Ejercicio 2 — Dividir por la mitad

Escribe `dividir(S)` que devuelva una tupla `(L, R)` con la mitad izquierda `S[:mid]` y la derecha `S[mid:]`, donde `mid = len(S) // 2`. Las listas devueltas deben ser copias.

In [15]:
def dividir(S):
    # Escribe tu solución aquí
    pass

In [16]:
verificar("par", dividir([27, 10, 12, 20]), ([27, 10], [12, 20]))
verificar("impar", dividir([1, 2, 3]), ([1], [2, 3]))
verificar("un elemento", dividir([5]), ([], [5]))

Incorrecto: par. Se esperaba ([27, 10], [12, 20]) y se obtuvo None
Incorrecto: impar. Se esperaba ([1], [2, 3]) y se obtuvo None
Incorrecto: un elemento. Se esperaba ([], [5]) y se obtuvo None


<details><summary>Ver solución</summary>

```python
def dividir(S):
    mid = len(S) // 2
    return S[:mid], S[mid:]
```

El corte de listas crea copias nuevas, igual que en `mergesort1`. Con longitud impar, la mitad derecha tiene un elemento más.
</details>

### Ejercicio 3 — Merge sort que devuelve una lista nueva

Escribe `ordenar_por_fusion(S)` recursiva que devuelva una lista nueva ordenada sin modificar `S`. Puedes usar la función `fusionar` del ejercicio 1 (se incluye definida en la celda de la solución del ejercicio).

In [17]:
def ordenar_por_fusion(S):
    # Escribe tu solución aquí
    pass

In [18]:
verificar("lista del PDF", ordenar_por_fusion([27, 10, 12, 20, 25, 13, 15, 22]), [10, 12, 13, 15, 20, 22, 25, 27])
verificar("vacía", ordenar_por_fusion([]), [])
verificar("no modifica la entrada", (lambda s: (ordenar_por_fusion(s), s))([3, 1, 2]), ([1, 2, 3], [3, 1, 2]))

Incorrecto: lista del PDF. Se esperaba [10, 12, 13, 15, 20, 22, 25, 27] y se obtuvo None
Incorrecto: vacía. Se esperaba [] y se obtuvo None
Incorrecto: no modifica la entrada. Se esperaba ([1, 2, 3], [3, 1, 2]) y se obtuvo (None, [3, 1, 2])


<details><summary>Ver solución</summary>

```python
def ordenar_por_fusion(S):
    if len(S) <= 1:
        return list(S)
    mid = len(S) // 2
    izq = ordenar_por_fusion(S[:mid])
    der = ordenar_por_fusion(S[mid:])
    return fusionar(izq, der)
```

Caso base: longitud 0 o 1. Caso recursivo: ordenar cada mitad y fusionar.
</details>

### Ejercicio 4 — Contar las ejecuciones de merge2

Escribe `contar_fusiones(S)` que devuelva cuántas veces se ejecuta la función de fusión al ordenar `S` con el esquema de `mergesort2` (rangos `low` y `high`, `mid = (low + high) // 2`). Pista: puedes contar sin ordenar, solo recorriendo los rangos.

In [19]:
def contar_fusiones(S):
    # Escribe tu solución aquí
    pass

In [20]:
verificar("10 elementos (PDF, pág. 108)", contar_fusiones([6, 2, 11, 7, 5, 4, 8, 16, 10, 3]), 9)
verificar("13 elementos (PDF, pág. 108)", contar_fusiones([6, 2, 11, 7, 5, 4, 8, 16, 10, 3, 1, 12, 9]), 12)
verificar("un elemento", contar_fusiones([1]), 0)

Incorrecto: 10 elementos (PDF, pág. 108). Se esperaba 9 y se obtuvo None
Incorrecto: 13 elementos (PDF, pág. 108). Se esperaba 12 y se obtuvo None
Incorrecto: un elemento. Se esperaba 0 y se obtuvo None


<details><summary>Ver solución</summary>

```python
def contar_fusiones(S):
    def contar(low, high):
        if low >= high:
            return 0
        mid = (low + high) // 2
        return contar(low, mid) + contar(mid + 1, high) + 1
    return contar(0, len(S) - 1)
```

Cada rango con más de un elemento produce una fusión más las de sus dos mitades. El resultado siempre es N - 1.
</details>

### Ejercicio 5 — Ordenar de mayor a menor

Escribe `ordenar_descendente(S)` que devuelva una lista nueva ordenada de mayor a menor usando merge sort (no uses `sorted`, `sort` ni `reverse`). Adapta la comparación de la fusión.

In [21]:
def ordenar_descendente(S):
    # Escribe tu solución aquí
    pass

In [22]:
verificar("caso general", ordenar_descendente([3, 9, 1, 7]), [9, 7, 3, 1])
verificar("repetidos", ordenar_descendente([2, 2, 1, 3]), [3, 2, 2, 1])
verificar("vacía", ordenar_descendente([]), [])

Incorrecto: caso general. Se esperaba [9, 7, 3, 1] y se obtuvo None
Incorrecto: repetidos. Se esperaba [3, 2, 2, 1] y se obtuvo None
Incorrecto: vacía. Se esperaba [] y se obtuvo None


<details><summary>Ver solución</summary>

```python
def ordenar_descendente(S):
    if len(S) <= 1:
        return list(S)
    mid = len(S) // 2
    izq = ordenar_descendente(S[:mid])
    der = ordenar_descendente(S[mid:])
    resultado = []
    i = j = 0
    while i < len(izq) and j < len(der):
        if izq[i] >= der[j]:
            resultado.append(izq[i])
            i += 1
        else:
            resultado.append(der[j])
            j += 1
    return resultado + izq[i:] + der[j:]
```

Basta con invertir la comparación: se toma primero el mayor de los dos primeros elementos.
</details>

### Ejercicio 6 — Fusionar N listas ordenadas

Problema de la pág. 112 del PDF. Escribe `fusionar_n(listas)` que reciba una lista de listas ordenadas y devuelva una única lista ordenada. Puedes reutilizar `fusionar` (del ejercicio 1).

In [23]:
def fusionar_n(listas):
    # Escribe tu solución aquí
    pass

In [24]:
verificar("tres listas", fusionar_n([[1, 5], [2, 6], [0, 9]]), [0, 1, 2, 5, 6, 9])
verificar("sin listas", fusionar_n([]), [])
verificar("con una vacía", fusionar_n([[], [4], [1, 8]]), [1, 4, 8])

Incorrecto: tres listas. Se esperaba [0, 1, 2, 5, 6, 9] y se obtuvo None
Incorrecto: sin listas. Se esperaba [] y se obtuvo None
Incorrecto: con una vacía. Se esperaba [1, 4, 8] y se obtuvo None


<details><summary>Ver solución</summary>

```python
def fusionar_n(listas):
    if not listas:
        return []
    pendientes = [list(x) for x in listas]
    while len(pendientes) > 1:
        siguiente = []
        for i in range(0, len(pendientes) - 1, 2):
            siguiente.append(fusionar(pendientes[i], pendientes[i + 1]))
        if len(pendientes) % 2 == 1:
            siguiente.append(pendientes[-1])
        pendientes = siguiente
    return pendientes[0]
```

Se fusionan de a pares por rondas; si sobra una lista, pasa tal cual a la siguiente ronda.
</details>

### Ejercicio 7 — Ordenamiento estable de pares

Escribe `ordenar_pares(pares)` que ordene una lista de pares `(clave, etiqueta)` por clave con merge sort **estable**: si dos claves son iguales, debe conservarse el orden original. Devuelve una lista nueva.

In [25]:
def ordenar_pares(pares):
    # Escribe tu solución aquí
    pass

In [26]:
verificar("estabilidad", ordenar_pares([(2, 'a'), (1, 'b'), (2, 'c'), (1, 'd')]), [(1, 'b'), (1, 'd'), (2, 'a'), (2, 'c')])
verificar("vacía", ordenar_pares([]), [])
verificar("ya ordenada", ordenar_pares([(1, 'x'), (2, 'y')]), [(1, 'x'), (2, 'y')])

Incorrecto: estabilidad. Se esperaba [(1, 'b'), (1, 'd'), (2, 'a'), (2, 'c')] y se obtuvo None
Incorrecto: vacía. Se esperaba [] y se obtuvo None
Incorrecto: ya ordenada. Se esperaba [(1, 'x'), (2, 'y')] y se obtuvo None


<details><summary>Ver solución</summary>

```python
def ordenar_pares(pares):
    if len(pares) <= 1:
        return list(pares)
    mid = len(pares) // 2
    izq = ordenar_pares(pares[:mid])
    der = ordenar_pares(pares[mid:])
    resultado = []
    i = j = 0
    while i < len(izq) and j < len(der):
        if izq[i][0] <= der[j][0]:   # <= mantiene primero el de la izquierda
            resultado.append(izq[i])
            i += 1
        else:
            resultado.append(der[j])
            j += 1
    return resultado + izq[i:] + der[j:]
```

La clave está en usar `<=` al comparar: ante un empate sale primero el elemento de la mitad izquierda, que era el que estaba antes.
</details>

### Ejercicio 8 — Contar inversiones con merge sort

Una inversión es un par de posiciones `i < j` con `S[i] > S[j]`. Escribe `contar_inversiones(S)` que devuelva el número de inversiones en tiempo O(N log N) aprovechando la fusión: cuando se toma un elemento de la derecha, se suma la cantidad de elementos que quedan en la izquierda. No modifiques `S`.

In [27]:
def contar_inversiones(S):
    # Escribe tu solución aquí
    pass

In [28]:
verificar("ordenada", contar_inversiones([1, 2, 3, 4]), 0)
verificar("invertida", contar_inversiones([4, 3, 2, 1]), 6)
verificar("mixta", contar_inversiones([2, 4, 1, 3, 5]), 3)

Incorrecto: ordenada. Se esperaba 0 y se obtuvo None
Incorrecto: invertida. Se esperaba 6 y se obtuvo None
Incorrecto: mixta. Se esperaba 3 y se obtuvo None


<details><summary>Ver solución</summary>

```python
def contar_inversiones(S):
    def ordenar(lista):
        if len(lista) <= 1:
            return lista, 0
        mid = len(lista) // 2
        izq, a = ordenar(lista[:mid])
        der, b = ordenar(lista[mid:])
        resultado, cruzadas = [], 0
        i = j = 0
        while i < len(izq) and j < len(der):
            if izq[i] <= der[j]:
                resultado.append(izq[i])
                i += 1
            else:
                resultado.append(der[j])
                j += 1
                cruzadas += len(izq) - i
        resultado.extend(izq[i:])
        resultado.extend(der[j:])
        return resultado, a + b + cruzadas
    return ordenar(list(S))[1]
```

Si `der[j]` es menor que `izq[i]`, también es menor que todo lo que queda en `izq` (está ordenada), y cada uno de esos pares es una inversión.
</details>

## Apuntes para estudiar

**Clave**
- El ordenamiento por fusión es divide y vencerás: dividir en dos mitades, ordenar cada una de forma recursiva y fusionar.
- Caso base: una lista de 0 o 1 elementos ya está ordenada.
- Fusionar dos listas ordenadas: comparar los primeros elementos, tomar el menor, repetir; al vaciarse una lista, copiar los sobrantes de la otra.
- `mergesort1(S)` crea listas `L` y `R` con cortes en cada llamada; usa memoria adicional del orden de 2N según el PDF.
- `mergesort2(S, low, high)` ordena por rangos de índices y fusiona con una lista auxiliar; el tamaño de `S` no cambia.
- Complejidad temporal O(N log N): unos log N niveles de recursión y N trabajo de fusión por nivel (análisis estándar, ampliación).
- Para N elementos se ejecutan N - 1 fusiones.

**Errores comunes**
- Olvidar copiar los elementos sobrantes tras el bucle principal de la fusión.
- Omitir el caso base y provocar recursión infinita.
- Llamar a `mergesort2` sin los índices `low` y `high`.
- Creer que dividir ordena; el orden se logra en la fusión.
- Usar `<` en lugar de `<=` al fusionar cuando importa la estabilidad (el `merge2` del PDF usa `<`).
- Olvidar que `S[:mid]` y `S[mid:]` crean copias: ordenar `L` no modifica `S`.

**Para el quiz**
- Complejidad: O(N log N), frente a O(N²) de burbuja, selección e inserción.
- Diferencia con los algoritmos O(N²): usa recursión y listas auxiliares, a cambio de más memoria.
- Memoria de `mergesort1`: 2N adicionales; se puede reducir a N.
- Veces que se ejecuta `merge2` con 10 elementos: 9; con 13: 12.
- Divide y vencerás se estudia de nuevo en la Unidad 31.

## Autoevaluación

**1. ¿Qué tres pasos componen el ordenamiento por fusión?**

<details><summary>Ver respuesta</summary>

Dividir la lista en dos mitades, ordenar cada mitad de forma recursiva (hasta listas de un elemento, ya ordenadas) y fusionar las dos mitades ordenadas en una lista ordenada. (PDF parte 1, págs. 76–82)
</details>

**2. ¿Por qué fusionar dos listas ordenadas es eficiente?**

<details><summary>Ver respuesta</summary>

Porque el menor de todos los elementos es siempre uno de los dos primeros, de modo que basta comparar los primeros elementos y avanzar; cada elemento se coloca una sola vez. El coste es lineal en el total de elementos. (PDF parte 1, págs. 83–91; el coste es ampliación)
</details>

**3. ¿En qué se diferencian `mergesort1` y `mergesort2`?**

<details><summary>Ver respuesta</summary>

`mergesort1` crea listas nuevas `L` y `R` con cortes en cada llamada (memoria adicional de 2N según el PDF); `mergesort2` ordena `S` en el sitio indicando rangos `low` y `high`, y fusiona con una lista auxiliar. (PDF parte 1, págs. 93–106)
</details>

**4. ¿Cuál es la complejidad temporal del ordenamiento por fusión y por qué?**

<details><summary>Ver respuesta</summary>

O(N log N): hay unos log N niveles de recursión y en cada nivel las fusiones procesan los N elementos. Es mejor que el O(N²) de burbuja, selección e inserción. (PDF parte 1, pág. 69; el razonamiento es ampliación)
</details>

**5. ¿Cuántas veces se ejecuta `merge2()` al ordenar una lista de N elementos con `mergesort2`?**

<details><summary>Ver respuesta</summary>

N - 1 veces. Con 10 elementos son 9 y con 13 son 12, que son las dos listas de la pregunta de la pág. 108. Cada fusión reduce en uno el número de listas, y se empieza con N listas de un elemento. (PDF parte 1, pág. 108)
</details>


---

[Índice de la semana](../semana-04.ipynb) | [Anterior](u27-burbuja-seleccion-insercion.ipynb) | [Siguiente](u29-ordenamiento-rapido.ipynb)